In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram
import plotly.express as px

In [ ]:
data_merged = pd.read_csv("../data/merged.csv")
data_merged.head()

In [ ]:
data_numeric = data_merged

In [ ]:
data = data_merged.copy().select_dtypes(include='number')
data = data.dropna()

In [ ]:
data.head()

In [ ]:
cls = PCA(n_components=10) 
pcs = cls.fit_transform(data) 
cls.components_
plt.bar([f"Axe {i+1}" for i in range(10)], cls.explained_variance_ratio_) 
plt.show()
np.cumsum(cls.explained_variance_ratio_)

In [ ]:

def add_labels(x, y, labels, ax=None):
    """Ajoute les étiquettes `labels` aux endroits définis par `x` et `y`."""

    if ax is None:
        ax = plt.gca()
    for x, y, label in zip(x, y, labels):
        ax.annotate(
            label, [x, y], xytext=(10, -5), textcoords="offset points",
        )

    return ax

In [ ]:
# Représentation dans le premier plan factoriel
plt.scatter(pcs[:, 0], pcs[:, 1]) 
# add_labels(pcs[:, 0], pcs[:, 1], data.index) 
plt.show()

In [ ]:
# Représentation dans le deuxième plan factoriel
plt.scatter(pcs[:, 1], pcs[:, 2]) 
plt.show()

In [ ]:
# Représentation dans le troisième plan factoriel
plt.scatter(pcs[:, 0], pcs[:, 2]) 
plt.show()

In [ ]:

modele_acp = cls

loadings_axes = modele_acp.components_
variables_names = data.columns

fig, ax = plt.subplots(figsize=(8, 8))

for i, col_name in enumerate(variables_names):
    x_val = loadings_axes[0, i]
    y_val = loadings_axes[1, i]
    

    ax.arrow(0, 0, x_val, y_val, head_width=0.025, head_length=0.025, color='red', alpha=0.6)
    

    ax.text(x_val * 1.15, y_val * 1.15, col_name, color='darkgreen', ha='center', va='center', fontsize=10)


cercle_unite = plt.Circle((0,0), 1, color='blue', fill=False, linestyle='--')
ax.add_artist(cercle_unite)


var_exp = modele_acp.explained_variance_ratio_
plt.xlabel(f"Axe Principal 1 ({var_exp[0]*100:.1f}%)", fontsize=11)
plt.ylabel(f"Axe Principal 2 ({var_exp[1]*100:.1f}%)", fontsize=11)

plt.title("Cercle des Corrélations (Variables - Student Alcohol)", fontsize=13, pad=15)
plt.grid(True, alpha=0.2)
plt.axis('equal')
plt.show()

In [ ]:

coefficients = cls.components_[:5, :]

axes_noms = []
for i in range(5):
    variance_pct = cls.explained_variance_ratio_[i] * 100
    axes_noms.append(f"Axe {i+1} ({variance_pct:.1f}%)")

tableau_pca = pd.DataFrame(
    coefficients.T, 
    columns=axes_noms, 
    index=data.columns
)
tableau_pca.round(3).style.background_gradient(cmap='coolwarm', axis=0)

In [ ]:
# Visualisation en 3D
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(pcs[:, 0], pcs[:, 1], pcs[:, 2])

ax.set_xlabel('Axe X (pcs 0)')
ax.set_ylabel('Axe Y (pcs 1)')
ax.set_zlabel('Axe Z (pcs 2)')

plt.show()

In [ ]:
# En version interactive 3d
df = pd.DataFrame(pcs[:, :3], columns=['pcs 0', 'pcs 1', 'pcs 2'])

fig = px.scatter_3d(df, x='pcs 0', y='pcs 1', z='pcs 2')
fig.show()

In [ ]:
# Transformation en distances (pour ensuite faire une AFTD dessus)
X = data.copy() 

# standardiser les données
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcul de la matrice de dissimilarité (avec la distance euclidienne)
matrice_dissimilarite = pairwise_distances(X_scaled, metric='euclidean')
df_distances = pd.DataFrame(matrice_dissimilarite, index=data.index, columns=data.index)

print(df_distances.head())

In [ ]:
# AFTD
aftd = MDS(n_components=2, dissimilarity='precomputed') 
dist = aftd.fit_transform(df_distances)
plt.scatter(*dist.T) 
add_labels(dist[:, 0], dist[:, 1], df_distances.index) 
plt.show()

In [ ]:
from scipy.spatial.distance import cdist
def plot_Shepard(mds_model, plot=True):
    """Affiche le diagramme de Shepard et retourne un couple contenant les
    dissimilarités originales et les distances apprises par le
    modèle.
    """

    assert isinstance(mds_model, MDS)

    # Inter-distances apprises
    dist = cdist(mds_model.embedding_, mds_model.embedding_)
    idxs = np.tril_indices_from(dist, k=-1)
    dist_mds = dist[idxs]

    # Inter-distances d'origine
    dist = mds_model.dissimilarity_matrix_
    dist_orig = dist[idxs]

    dists = np.column_stack((dist_orig, dist_mds))

    if plot:
        f, ax = plt.subplots()
        range = [dists.min(), dists.max()]
        ax.plot(range, range, 'r--')
        ax.scatter(*dists.T)
        ax.set_xlabel('Dissimilarités')
        ax.set_ylabel('Distances')

    return (*dists.T,)

In [ ]:
aftd = MDS(n_components=2, dissimilarity='precomputed') 
dist = aftd.fit_transform(df_distances) 
plot_Shepard(aftd) 
plt.show()

In [ ]:
aftd = MDS(n_components=3, dissimilarity='precomputed') 
dist = aftd.fit_transform(df_distances) 
plot_Shepard(aftd) 
plt.show()

In [ ]:
aftd = MDS(n_components=4, dissimilarity='precomputed') 
dist = aftd.fit_transform(df_distances) 
plot_Shepard(aftd) 
plt.show()

In [ ]:
aftd = MDS(n_components=5, dissimilarity='precomputed') 
dist = aftd.fit_transform(df_distances) 
plot_Shepard(aftd) 
plt.show()

In [ ]:
stress_shepard = aftd.stress_
print(f"Stress théorique = {stress_shepard}")
diss, dist = plot_Shepard(aftd, plot=False)
stress_exp = np.sum((diss - dist)**2)
print(f"Stress théorique = {stress_exp}")

In [ ]:
def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram

    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack([model.children_, model.distances_,
                                      counts]).astype(float)

    # Plot the corresponding dendrogram
    default_kwargs = dict(leaf_font_size=10)
    default_kwargs.update(kwargs or {})
        
    dendrogram(linkage_matrix, **default_kwargs)


In [ ]:
# CAH
cls = AgglomerativeClustering( metric="euclidean", linkage="ward", distance_threshold=0,n_clusters=None)
cls.fit(data)
plot_dendrogram(cls)
plt.show()